# Tic-Tac-Toe Exploration

This notebook is now an experimentation layer over the `game_sandbox` package. The package is the source of truth for rules, agents, metrics, reporting, and matchup execution.

In [ ]:
from game_sandbox.agents.alphabeta import (
    AlphaBetaAgent,
    alphabeta_agent,
    alphabeta_o_wrapper,
    alphabeta_search,
    alphabeta_x_wrapper,
)
from game_sandbox.agents import (
    human_agent,
    minimax_o_wrapper,
    minimax_x_wrapper,
    random_wrapper,
)
from game_sandbox.games.tic_tac_toe import currentPlayer, rule_engine, status
from game_sandbox.observability import print_match_metrics
from game_sandbox.runner import matchup


## One Match With Metrics

In [ ]:
metrics = matchup(
    rule_engine(),
    minimax_x_wrapper,
    random_wrapper,
    "Minimax",
    "Random",
)

print_match_metrics(metrics)


## Alpha-Beta Match With Metrics

In [ ]:
metrics = matchup(
    rule_engine(),
    alphabeta_x_wrapper,
    random_wrapper,
    "Alpha-Beta",
    "Random",
)

print_match_metrics(metrics)


## Standard Experiment Matrix

In [ ]:
experiments = [
    ("Random", random_wrapper, "Random", random_wrapper),
    ("Minimax", minimax_x_wrapper, "Random", random_wrapper),
    ("Alpha-Beta", alphabeta_x_wrapper, "Random", random_wrapper),
    ("Random", random_wrapper, "Minimax", minimax_o_wrapper),
    ("Random", random_wrapper, "Alpha-Beta", alphabeta_o_wrapper),
    ("Minimax", minimax_x_wrapper, "Minimax", minimax_o_wrapper),
    ("Alpha-Beta", alphabeta_x_wrapper, "Alpha-Beta", alphabeta_o_wrapper),
    ("Minimax", minimax_x_wrapper, "Alpha-Beta", alphabeta_o_wrapper),
    ("Alpha-Beta", alphabeta_x_wrapper, "Minimax", minimax_o_wrapper),
]

for x_name, x_agent, o_name, o_agent in experiments:
    metrics = matchup(rule_engine(), x_agent, o_agent, x_name, o_name)
    print(f"{x_name} vs {o_name}: {metrics.winner.name} in {metrics.no_of_moves} moves")


## Minimax vs Alpha-Beta Search Cost

In [ ]:
def summarize_search(metrics):
    totals = {
        "nodes": 0,
        "terminal_nodes": 0,
        "branches": 0,
        "deep_copies": 0,
        "pruning_cutoffs": 0,
        "max_depth": 0,
    }
    for decision in metrics.decisions:
        search = decision.search_metrics
        totals["nodes"] += search.nodes_explored
        totals["terminal_nodes"] += search.terminal_nodes
        totals["branches"] += search.branches_considered
        totals["deep_copies"] += search.deep_copies
        totals["pruning_cutoffs"] += search.pruning_cutoffs
        totals["max_depth"] = max(totals["max_depth"], search.max_depth)
    return totals


search_cost_experiments = [
    ("Minimax", minimax_x_wrapper, minimax_o_wrapper),
    ("Alpha-Beta", alphabeta_x_wrapper, alphabeta_o_wrapper),
]

for agent_name, x_agent, o_agent in search_cost_experiments:
    metrics = matchup(rule_engine(), x_agent, o_agent, agent_name, agent_name)
    totals = summarize_search(metrics)
    print(
        f"{agent_name}: winner={metrics.winner.name}, moves={metrics.no_of_moves}, "
        f"nodes={totals['nodes']:,}, branches={totals['branches']:,}, "
        f"cutoffs={totals['pruning_cutoffs']:,}, max_depth={totals['max_depth']}"
    )


## Human Interaction

Run this cell to play as X against the Minimax O agent.

In [ ]:
metrics = matchup(
    rule_engine(),
    human_agent,
    minimax_o_wrapper,
    "Human",
    "Minimax",
)

print_match_metrics(metrics)
